# Decorators

A decorator is a function that manipulates another function or class. It is a very useful feature in the Python language.

In these practical exercises, we will learn how to use decorators.


## Function Decorators: Functions That Manipulate Functions

Write a function `squared` that takes a function as input and returns that same function with its result squared:

- the original function always takes two arguments
- the original function always produces a number
- the squared function takes **a function** as input and produces one as output

In [ ]:
def add(a, b):
  return a + b


def sub(a, b):
  return a - b


def mul(a, b):
  return a * b


def squared(function):
  return function  # Your code here


print(squared(add)(2, 2))  # Should be 16
print(squared(sub)(10, 5))  # Should be 25
print(squared(mul)(3, 4))  # Should be 144

### Solution

In [ ]:
def squared(function):
  def new_function(a, b):
    return function(a, b) ** 2
  return new_function


print(squared(add)(2, 2))
print(squared(sub)(10, 5))
print(squared(mul)(3, 4))

## Order of Stacked Decorators

The two decorators below print a message before and after calling the function they decorate, and wrap its result in an HTML tag.

Without running any code, write down, in order, the lines that `print(greet("Ada"))` will print. Then check your answer by running both cells.

In [ ]:
import functools


def bold(function):
    @functools.wraps(function)
    def new_function(*args, **kwargs):
        print("bold: before")
        result = function(*args, **kwargs)
        print("bold: after")
        return f"<b>{result}</b>"

    return new_function


def italic(function):
    @functools.wraps(function)
    def new_function(*args, **kwargs):
        print("italic: before")
        result = function(*args, **kwargs)
        print("italic: after")
        return f"<i>{result}</i>"

    return new_function


@bold
@italic
def greet(name):
    print("greet")
    return f"Hello {name}"

In [ ]:
print(greet("Ada"))

*Your answer*

### Solution

```text
bold: before
italic: before
greet
italic: after
bold: after
<b><i>Hello Ada</i></b>
```

`@bold @italic def greet` is equivalent to `greet = bold(italic(greet))`: `bold` wraps `italic`, which wraps `greet`. The call goes through the layers from the outside in, and the result comes back out in reverse order.

## Measuring a Function's Execution Time

Create a `timeit` decorator that displays a function's execution time. You can use the [`time.perf_counter`](https://docs.python.org/3/library/time.html#time.perf_counter) function

In [ ]:
# Your code here

### Solution

In [ ]:
import time


def timeit(f):
  def new_f(*args, **kwargs):
    start = time.perf_counter()
    result = f(*args, **kwargs)
    print(f"Time elapsed during '{f.__name__}' call: {time.perf_counter() - start}s")
    return result
  return new_f


@timeit
def add(a, b):
  return a + b


add(1, 2)

## Class Registry

It's often useful to register classes or functions in a data structure (a registry), for instance to offer a plugin system.

Create a class decorator that registers each class it's applied to in a dictionary, under the key `cls.__name__`.

In [ ]:
# Your code here

In addition to adding the decorator to the classes you want to register, what important step is necessary for the registration to occur?


*Your answer*

### Solution

This pattern, called *Registry*, lets a library register client code so it can run it later at the right time.

It often happens in three steps:

1. Providing a decorator to register a function or a class in the client code
2. Using the decorator on the client side
3. Using the registered elements on the library side

For instance, for step 1.:

In [ ]:
from typing import Protocol


class SupportsProcess(Protocol):
  def process(self) -> None: ...


_plugins = {}


def register[T: SupportsProcess](cls: type[T]) -> type[T]:
  _plugins[cls.__name__] = cls
  return cls

Then for step 2.:

In [ ]:
@register
class CustomClientProcessor:
  def process(self) -> None:
    print("Print from the client's custom process method")

Careful: the module containing the class must be loaded (otherwise the decorator won't run and the class won't be registered).

Finally, for step 3.:

In [ ]:
for name, cls in _plugins.items():
  print(f"Calling the process method of {name}")
  cls().process()

## Defining a Decorator With an Argument

Take the previous exercise, but this time the decorator must take a `name` argument to use as the registry key instead of `cls.__name__`.

In [ ]:
# Your code here

### Solution

In [ ]:
from collections.abc import Callable

_plugins = {}


def register[T](name: str) -> Callable[[type[T]], type[T]]:
    def decorator(cls: type[T]) -> type[T]:
        _plugins[name] = cls
        return cls

    return decorator


@register("csv")
class CsvExporter:
    pass


_plugins

## Using [`functools.wraps`](https://docs.python.org/3/library/functools.html#functools.wraps)

Try manipulating the functions modified in the rest of the lab and finding their name (`__name__` attribute) or their doc (`__doc__` attribute). What do you notice?

*Your answer*

Create a `timeit` decorator that takes a `unit` argument (`"s"` or `"ms"`), the unit in which to display the execution time, then use `functools.wraps` to solve this issue.

In [ ]:
# Your code here

### Solution

In [ ]:
import functools
import time


def timeit(unit="s"):
  def decorator(function):
    @functools.wraps(function)
    def new_f(*args, **kwargs):
      start = time.perf_counter()
      result = function(*args, **kwargs)
      duration = time.perf_counter() - start
      if unit == "ms":
        duration *= 1000
      elif unit != "s":
        raise ValueError("Can only use s or ms as unit argument")
      print(f"Time elapsed during '{function.__name__}' call: "
            f"{duration}{unit}")
      return result
    return new_f
  return decorator


@timeit(unit="ms")
def add(a, b):
  """Add two integers."""
  return a + b


print(add.__name__, add.__doc__)

## Caching Results

Write a `memoize` decorator that stores the result of each call in a dictionary, keyed by its arguments, and returns it directly when the function is called again with the same arguments.

Apply it to `fibonacci`, then use your `timeit` decorator to compare how long `fibonacci(32)` takes without a cache, with `memoize`, and with [`functools.cache`](https://docs.python.org/3/library/functools.html#functools.cache).

In [ ]:
def fibonacci(n: int) -> int:
    return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)

In [ ]:
# Your code here

### Solution

In [ ]:
import functools


def memoize(function):
    cache = {}

    @functools.wraps(function)
    def new_function(*args):
        if args not in cache:
            cache[args] = function(*args)
        return cache[args]

    return new_function


def fibonacci_plain(n: int) -> int:
    return n if n < 2 else fibonacci_plain(n - 1) + fibonacci_plain(n - 2)


@memoize
def fibonacci(n: int) -> int:
    return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)


@functools.cache
def fibonacci_cache(n: int) -> int:
    return n if n < 2 else fibonacci_cache(n - 1) + fibonacci_cache(n - 2)


for f in (fibonacci_plain, fibonacci, fibonacci_cache):
    timeit(unit="ms")(f)(32)

Without a cache, `fibonacci(32)` recomputes the same values millions of times; with one, each value is computed once. `functools.cache` does the same as `memoize`, also handles keyword arguments, and provides `cache_info()` and `cache_clear()`.